In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-obp qiskit-addon-utils rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# การย้อนกลับตัวดำเนินการ (OBP) สำหรับการประมาณค่าความคาดหวัง

*ประมาณเวลาการใช้งาน: 16 นาทีบนโปรเซสเซอร์ Eagle r3 (หมายเหตุ: นี่เป็นเพียงการประมาณการ เวลาจริงอาจแตกต่างกันได้)*

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.primitives import StatevectorEstimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import LieTrotter

from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth, combine_slices
from qiskit_addon_obp.utils.simplify import OperatorBudget
from qiskit_addon_obp import backpropagate
from qiskit_addon_obp.utils.truncating import setup_budget

from rustworkx.visualization import graphviz_draw

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2, EstimatorOptions

## พื้นหลัง
การย้อนกลับตัวดำเนินการ (Operator backpropagation) คือเทคนิคที่ดูดซับการดำเนินการจากส่วนท้ายของ Circuit ควอนตัมเข้าสู่ observable ที่วัดได้ ซึ่งโดยทั่วไปจะลดความลึกของ Circuit แลกกับจำนวน term ที่เพิ่มขึ้นใน observable เป้าหมายคือย้อนกลับ Circuit ให้ได้มากที่สุดเท่าที่จะทำได้โดยไม่ให้ observable ขยายใหญ่เกินไป การนำไปใช้งานบน Qiskit มีอยู่ใน OBP Qiskit addon โดยสามารถดูรายละเอียดเพิ่มเติมได้ที่ [เอกสาร](/guides/qiskit-addons-obp) พร้อม [ตัวอย่างง่าย ๆ](/guides/qiskit-addons-obp-get-started) สำหรับการเริ่มต้น

พิจารณา Circuit ตัวอย่างที่ต้องวัด observable $O = \sum_P c_P P$ โดยที่ $P$ คือ Pauli และ $c_P$ คือสัมประสิทธิ์ กำหนดให้ Circuit แทนด้วย unitary เดี่ยว $U$ ซึ่งสามารถแบ่งแบบ logical ได้เป็น $U = U_C U_Q$ ดังแสดงในรูปด้านล่าง

![Circuit diagram showing Uq followed by Uc](../docs/images/tutorials/improving-estimation-of-expectation-values-with-operator-backpropagation/logical-partitioning.avif)

การย้อนกลับตัวดำเนินการดูดซับ unitary $U_C$ เข้าสู่ observable โดยวิวัฒนาการมันเป็น $O' = U_C^{\dagger}OU_C = \sum_P c_P U_C^{\dagger}PU_C$ พูดอีกนัยหนึ่งคือ ส่วนหนึ่งของการคำนวณดำเนินการแบบ classical ผ่านการวิวัฒนาการของ observable จาก $O$ ไปเป็น $O'$ ปัญหาเดิมสามารถกำหนดใหม่ได้เป็นการวัด observable $O'$ สำหรับ Circuit ใหม่ที่มีความลึกน้อยลง ซึ่งมี unitary เป็น $U_Q$

Unitary $U_C$ แทนด้วยจำนวน slice $U_C = U_S U_{S-1}...U_2U_1$ มีหลายวิธีในการกำหนด slice ตัวอย่างเช่น ใน Circuit ตัวอย่างข้างต้น แต่ละเลเยอร์ของ $R_{zz}$ และแต่ละเลเยอร์ของ Gate $R_x$ สามารถถือเป็น slice ได้ การย้อนกลับเกี่ยวข้องกับการคำนวณ $O' = \Pi_{s=1}^S \sum_P c_P U_s^{\dagger} P U_s$ แบบ classical โดย slice แต่ละอัน $U_s$ สามารถแทนได้เป็น $U_s = exp(\frac{-i\theta_s P_s}{2})$ โดยที่ $P_s$ คือ Pauli ขนาด $n$ Qubit และ $\theta_s$ คือสเกลาร์ สามารถตรวจสอบได้ง่ายว่า

$$
U_s^{\dagger} P U_s = P \qquad \text{if} ~[P,P_s] = 0,
$$
$$
U_s^{\dagger} P U_s = \qquad cos(\theta_s)P + i sin(\theta_s)P_sP \qquad \text{if} ~{P,P_s} = 0
$$

ในตัวอย่างข้างต้น ถ้า ${P,P_s} = 0$ เราต้องรัน Circuit ควอนตัมสอง Circuit แทนที่จะเป็นหนึ่ง Circuit เพื่อคำนวณค่าความคาดหวัง ดังนั้น การย้อนกลับอาจเพิ่มจำนวน term ใน observable ทำให้ต้องรัน Circuit มากขึ้น วิธีหนึ่งที่จะอนุญาตให้ย้อนกลับลึกกว่าเข้าไปใน Circuit โดยไม่ให้ตัวดำเนินการขยายใหญ่เกินไปคือการตัดทอน term ที่มีสัมประสิทธิ์เล็ก แทนที่จะเพิ่มลงใน operator ตัวอย่างเช่น ในตัวอย่างข้างต้น อาจเลือกตัด term ที่เกี่ยวกับ $P_sP$ ออก หากว่า $\theta_s$ เล็กพอ การตัดทอน term อาจส่งผลให้ต้องรัน Circuit ควอนตัมน้อยลง แต่จะเกิดข้อผิดพลาดบางส่วนในการคำนวณค่าความคาดหวังสุดท้ายซึ่งสัดส่วนกับขนาดของสัมประสิทธิ์ของ term ที่ถูกตัดออก

บทเรียนนี้นำ [รูปแบบ Qiskit](/guides/intro-to-patterns) มาใช้จำลองพลศาสตร์ควอนตัมของโซ่สปิน Heisenberg โดยใช้ <a href="https://github.com/Qiskit/qiskit-addon-obp">qiskit-addon-obp</a>

## ข้อกำหนดเบื้องต้น
ก่อนเริ่มบทเรียนนี้ ให้แน่ใจว่าได้ติดตั้งสิ่งต่อไปนี้แล้ว:

- Qiskit SDK v1.2 หรือใหม่กว่า (`pip install qiskit`)
- Qiskit Runtime v0.28 หรือใหม่กว่า (`pip install qiskit-ibm-runtime`)
- OBP Qiskit addon (`pip install qiskit-addon-obp`)
- Qiskit addon utils (`pip install qiskit-addon-utils`)

## การตั้งค่า

In [2]:
num_qubits = 10
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)
graphviz_draw(coupling_map.graph, method="circo")

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

## ส่วนที่ I: โซ่สปิน Heisenberg ขนาดเล็ก
### ขั้นตอนที่ 1: แมป input แบบ classical ไปสู่ปัญหาควอนตัม
#### แมปการวิวัฒนาการตามเวลาของโมเดลควอนตัม Heisenberg ไปสู่การทดลองควอนตัม
แพ็กเกจ [qiskit_addon_utils](https://github.com/Qiskit/qiskit-addon-utils) มีฟังก์ชันที่สามารถนำกลับมาใช้ใหม่ได้สำหรับวัตถุประสงค์ต่าง ๆ

โมดูล [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) มีฟังก์ชันสำหรับสร้าง Hamiltonian แบบ Heisenberg บนกราฟการเชื่อมต่อที่กำหนด
กราฟนี้สามารถเป็น [rustworkx.PyGraph](https://www.rustworkx.org/apiref/rustworkx.PyGraph.html) หรือ [CouplingMap](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.CouplingMap) ซึ่งใช้งานได้ง่ายในขั้นตอนงานที่เน้น Qiskit

ในส่วนต่อไปนี้ เราจะสร้าง `CouplingMap` แบบโซ่เชิงเส้นที่มี 10 Qubit

In [3]:
# Get a qubit operator describing the Heisenberg XYZ model
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII', 'IIIIIIIIIX', 'IIIIIIIIIY', 'IIIIIIIIIZ', 'IIIIIIIIXI', 'IIIIIIIIYI', 'IIIIIIIIZI', 'IIIIIIIXII', 'IIIIIIIYII', 'IIIIIIIZII', 'IIIIIIXIII', 'IIIIIIYIII', 'IIIIIIZIII', 'IIIIIXIIII', 'IIIIIYIIII', 'IIIIIZIIII', 'IIIIXIIIII', 'IIIIYIIIII', 'IIIIZIIIII', 'IIIXIIIIII', 'IIIYIIIIII', 'IIIZIIIIII', 'IIXIIIIIII', 'IIYIIIIIII', 'IIZIIIIIII', 'IXIIIIIIII', 'IYIIIIIIII', 'IZIIIIIIII', 'XIIIIIIIII', 'YIIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/de8ce35e-a2c5-474f-adb9-88c9afb2bd76-0.avif)

ต่อไป เราจะสร้าง Pauli operator ที่จำลอง Hamiltonian XYZ แบบ Heisenberg

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{(j,k)\in E} (J_{x} \sigma_j^{x} \sigma_{k}^{x} + J_{y} \sigma_j^{y} \sigma_{k}^{y} + J_{z} \sigma_j^{z} \sigma_{k}^{z}) + \sum_{j\in V} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z})}
$$

โดยที่ $G(V,E)$ คือกราฟของ coupling map ที่ให้มา

In [4]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=2),
)
circuit.draw("mpl", style="iqp", fold=-1)

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum hardware execution

#### Create circuit slices to backpropagate

The `backpropagate` function backpropagates entire circuit slices at a time. Therefore, the choice of slicing can have an impact on how well backpropagation performs for a given problem. Here, we will group gates of the same type into slices using the [`slice_by_depth`](/docs/api/qiskit-addon-utils/slicing#slice_by_depth) function.

For a more detailed discussion on circuit slicing, check out this [how-to guide](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) of the [`qiskit-addon-utils`](https://github.com/Qiskit/qiskit-addon-utils) package.

In [5]:
slices = slice_by_depth(circuit, max_slice_depth=1)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 18 slices.


จาก qubit operator เราสามารถสร้าง Circuit ควอนตัมที่จำลองการวิวัฒนาการตามเวลาของมันได้
อีกครั้ง โมดูล [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) มีฟังก์ชันที่สะดวกสำหรับทำสิ่งนี้:

In [6]:
op_budget = OperatorBudget(max_qwc_groups=8)

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/1d68f197-ffa4-49de-9fe8-243b1facbd00-0.avif)

### ขั้นตอนที่ 2: ปรับปัญหาให้เหมาะสมสำหรับการรันบน hardware ควอนตัม
#### สร้าง slice ของ Circuit สำหรับการย้อนกลับ
จำไว้ว่าฟังก์ชัน ``backpropagate`` จะย้อนกลับ Circuit ทีละ slice ดังนั้นการเลือกวิธีตัด slice อาจมีผลต่อประสิทธิภาพของการย้อนกลับสำหรับปัญหาที่กำหนด ที่นี่เราจะจัดกลุ่ม Gate ประเภทเดียวกันเป็น slice โดยใช้ฟังก์ชัน [slice_by_gate_types](https://docs.quantum.ibm.com/api/qiskit-addon-utils/slicing#slice_by_gate_types)

สำหรับการอภิปรายเพิ่มเติมเกี่ยวกับการตัด Circuit เป็น slice ดู [how-to guide](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) ของแพ็กเกจ [qiskit-addon-utils](https://github.com/Qiskit/qiskit-addon-utils)

In [7]:
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits=num_qubits,
)
observable

SparsePauliOp(['IIIIIIIIIZ', 'IIIIIIIIZI', 'IIIIIIIZII', 'IIIIIIZIII', 'IIIIIZIIII', 'IIIIZIIIII', 'IIIZIIIIII', 'IIZIIIIIII', 'IZIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j,
 0.1+0.j, 0.1+0.j])

Below you will see that we backpropagated six slices, and the terms were combined into six and not eight groups. This implies that backpropagating one more slice would cause the number of Pauli groups to exceed eight. We can verify that this is the case by inspecting the returned metadata. Also note that in this portion the circuit transformation is exact.  That is, no terms of the new observable $O’$ were truncated. The backpropagated circuit and the backpropagated operator give the exact outcome as the original circuit and operator.

In [8]:
# Backpropagate slices onto the observable
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
# Recombine the slices remaining after backpropagation
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into {len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit.draw("mpl", fold=-1, scale=0.6)

Backpropagated 6 slices.
New observable has 60 terms, which can be combined into 6 groups.
Note that backpropagating one more slice would result in 114 terms across 12 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/ee8fd385-1.avif" alt="Output of the previous code cell" />

#### จำกัดขนาดที่ operator อาจเติบโตในระหว่างการย้อนกลับ
ในระหว่างการย้อนกลับ จำนวน term ใน operator โดยทั่วไปจะเข้าใกล้ $4^N$ อย่างรวดเร็ว โดยที่ $N$ คือจำนวน Qubit เมื่อ term สอง term ใน operator ไม่ commute กันแบบ qubit-wise เราต้องการ Circuit แยกกันเพื่อหาค่าความคาดหวังที่สอดคล้องกัน ตัวอย่างเช่น ถ้ามี observable 2 Qubit $O = 0.1 XX + 0.3 IZ - 0.5 IX$ เนื่องจาก $[XX,IX] = 0$ การวัดในฐานเดียวก็เพียงพอเพื่อคำนวณค่าความคาดหวังสำหรับ term สองนั้น อย่างไรก็ตาม $IZ$ anti-commute กับอีกสอง term ดังนั้นเราต้องการการวัดฐานแยกต่างหากเพื่อคำนวณค่าความคาดหวังของ $IZ$ พูดอีกนัยหนึ่งคือ เราต้องการสอง Circuit แทนที่จะเป็นหนึ่ง Circuit เพื่อคำนวณ $\langle O \rangle$ เมื่อจำนวน term ใน operator เพิ่มขึ้น จำนวนการรัน Circuit ที่ต้องการก็อาจเพิ่มขึ้นด้วย

ขนาดของ operator สามารถจำกัดได้โดยระบุ kwarg ``operator_budget`` ของฟังก์ชัน ``backpropagate`` ซึ่งรับ instance ของ [OperatorBudget](https://docs.quantum.ibm.com/api/qiskit-addon-obp/utils-simplify#operatorbudget)

เพื่อควบคุมปริมาณทรัพยากรเพิ่มเติม (เวลา) ที่จัดสรร เราจำกัดจำนวน qubit-wise commuting Pauli group สูงสุดที่ observable ที่ย้อนกลับแล้วจะมีได้ ที่นี่เราระบุว่าการย้อนกลับควรหยุดเมื่อจำนวน qubit-wise commuting Pauli group ใน operator เกิน 8

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(backend)

<IBMBackend('ibm_kingston')>


In [10]:
pm_basis = generate_preset_pass_manager(
    optimization_level=3, basis_gates=backend.configuration().basis_gates
)
isa_circuit = pm_basis.run(circuit)
isa_bp_circuit = pm_basis.run(bp_circuit)

### Step 3: Execute using Qiskit primitives

First, we create two [Primitive Unified Blocs](/docs/api/qiskit/primitives) (PUBs) corresponding to the original circuit, and the backpropagated circuit. Then we execute the pubs on an ideal Estimator to obtain the expectation values.

In [11]:
pubs = [(isa_circuit, observable), (isa_bp_circuit, bp_obs)]

In [12]:
rng = np.random.default_rng()
estimator = StatevectorEstimator(seed=rng)
job = estimator.run(pubs)

### Step 4: Post-process and return result in desired classical format

Now we obtain the expectation values of the original and backpropagated circuits.

In [13]:
primitive_result = job.result()
circuit_expval = primitive_result[0].data.evs.item()
bp_circuit_expval = primitive_result[1].data.evs.item()

In [14]:
methods = [
    "No backpropagation",
    "Backpropagation",
]
values = [circuit_expval, bp_circuit_expval]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylim([0.6, 0.92])
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

สังเกตว่าการจัดสรร error ขนาด `5e-3` ต่อ slice สำหรับการตัดทอน ทำให้เราสามารถลบ 1 slice เพิ่มเติมจาก Circuit ในขณะที่ยังอยู่ภายใน budget เดิมที่กำหนดไว้สำหรับ commuting Pauli group จำนวนแปดกลุ่มใน observable โดยค่าเริ่มต้น `backpropagate` ใช้ L1 norm ของสัมประสิทธิ์ที่ถูกตัดทอนเพื่อจำกัดข้อผิดพลาดรวมที่เกิดจากการตัดทอน สำหรับตัวเลือกอื่น ๆ ดู [how-to guide เกี่ยวกับการระบุ p_norm](https://qiskit.github.io/qiskit-addon-obp/how_tos/bound_error_using_p_norm.html)

ในตัวอย่างนี้ที่เราย้อนกลับ 7 slice ข้อผิดพลาดจากการตัดทอนรวมไม่ควรเกิน ``(5e-3 error/slice) * (7 slices) = 3.5e-2``
สำหรับการอภิปรายเพิ่มเติมเกี่ยวกับการกระจาย error budget ให้กับ slice ดู [how-to guide นี้](https://qiskit.github.io/qiskit-addon-obp/how_tos/truncate_operator_terms.html)

In [15]:
num_qubits = 50
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)

# Generate a time evolution circuit for the Hamiltonian
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=4),
)

# Define the observable to measure
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits,
)

slices = slice_by_depth(circuit, max_slice_depth=1)

# Define the maximum number of qwc groups allowed in the backpropagated observable, and the truncation error budget
op_budget = OperatorBudget(max_qwc_groups=15)
truncation_error_budget = setup_budget(
    max_error_total=0.03, max_error_per_slice=0.005
)

# First backpropagation without truncation
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
bp_circuit = combine_slices(remaining_slices)

# Now backpropagate with truncation, using the same operator budget and the defined truncation error budget
bp_obs_trunc, remaining_slices_trunc, metadata = backpropagate(
    observable,
    slices,
    operator_budget=op_budget,
    truncation_error_budget=truncation_error_budget,
)
bp_circuit_trunc = combine_slices(
    remaining_slices_trunc, include_barriers=False
)

# Now we transpile the original circuit and the two backpropagated circuits, and apply the layout to the corresponding observables
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

isa_circuit = pm.run(circuit)
isa_bp_circuit = pm.run(bp_circuit)
isa_bp_circuit_trunc = pm.run(bp_circuit_trunc)

isa_observable = observable.apply_layout(isa_circuit.layout)
isa_bp_observable = bp_obs.apply_layout(isa_bp_circuit.layout)
isa_bp_observable_trunc = bp_obs_trunc.apply_layout(
    isa_bp_circuit_trunc.layout
)

# Compare the 2-qubit depth of each transpiled circuit to see how much depth backpropagation saved
print(
    f"2-qubit depth without backpropagation: {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation: {isa_bp_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation and truncation: {isa_bp_circuit_trunc.depth(lambda x: x.operation.num_qubits == 2)}"
)

pubs = [
    (isa_circuit, isa_observable),
    (isa_bp_circuit, isa_bp_observable),
    (isa_bp_circuit_trunc, isa_bp_observable_trunc),
]

# Now we instantiate the Estimator primitive for the hardware with ZNE and measurement error mitigation
# and compute the three circuits and observables
options = EstimatorOptions()
options.default_precision = 0.01
options.resilience_level = 2
options.resilience.zne.noise_factors = [1, 1.2, 1.4]
options.resilience.zne.extrapolator = ["linear"]
estimator = EstimatorV2(mode=backend, options=options)

estimator.options.environment.job_tags = ["TUT_OBP"]
job = estimator.run(pubs)

# Retrieve the results and the standard deviations
result_no_bp = job.result()[0].data.evs.item()
result_bp = job.result()[1].data.evs.item()
result_bp_trunc = job.result()[2].data.evs.item()

std_no_bp = job.result()[0].data.stds.item()
std_bp = job.result()[1].data.stds.item()
std_bp_trunc = job.result()[2].data.stds.item()

2-qubit depth without backpropagation: 24
2-qubit depth with backpropagation: 20
2-qubit depth with backpropagation and truncation: 18


In [16]:
print(f"Expectation value without backpropagation: {result_no_bp}")
print(f"Backpropagated expectation value: {result_bp}")
print(f"Backpropagated expectation value with truncation: {result_bp_trunc}")

Expectation value without backpropagation: 0.9543907942381811
Backpropagated expectation value: 0.9445337385406468
Backpropagated expectation value with truncation: 0.934050286970965


In [17]:
# Plot the results
methods = [
    "No backpropagation",
    "Backpropagation",
    "Backpropagation w/ truncation",
]
values = [result_no_bp, result_bp, result_bp_trunc]
error_bars = [std_no_bp, std_bp, std_bp_trunc]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.errorbar(methods, values, yerr=error_bars, fmt="o", color="r", capsize=5)
plt.axhline(0.89)
ax.set_ylim([0.8, 0.98])
plt.text(0.25, 0.895, "Exact result")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/37834c72-1.avif" alt="Output of the previous code cell" />

## Next steps
If you found this work interesting, you might be interested in the following material:
<Admonition type="tip" title="Recommendations">
- [Approximate quantum compilation for time evolution circuits](/docs/tutorials/approximate-quantum-compilation-for-time-evolution)
- [Multi-product formulas to reduce Trotter error](/docs/tutorials/multi-product-formula)
- [`pauli-prop`](https://github.com/Qiskit/pauli-prop), a Rust-accelerated package for Pauli propagation, with [tutorials](https://github.com/Qiskit/pauli-prop/tree/main/docs/tutorials) covering OBP, classical expectation-value estimation, and noisy simulation
</Admonition>